# drfn-nllb: the grounded root memory inside a pretrained translator

Host: NLLB-200-distilled-600M (Ge'ez is not one of its languages; source tag tir_Ethi).
Same cleaned AGE data and analyser labels as the small-model grid.

| run | what |
|---|---|
| zeroshot | NLLB as released |
| ft | fine-tuned |
| aux | fine-tuned + root/pattern side heads |
| gmm | fine-tuned + grounded memory after encoder layers 4 and 8 (starts as an exact no-op) |

Seeds 11 and 22 for ft / aux / gmm. Decoding: beam 5, no-repeat 4-gram.

In [ ]:
!pip install -q sacrebleu sentencepiece
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_nllb.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
from huggingface_hub import snapshot_download
print(snapshot_download('facebook/nllb-200-distilled-600M'))

In [ ]:
print('exit', subprocess.call(['bash', 'run_nllb.sh']))

In [ ]:
import json
for p in sorted(glob.glob('/kaggle/working/nllb/*/eval.json')):
    r = json.load(open(p)); d = r['dev_beam5_nr4']; pr = r.get('probe') or {}; sw = r.get('slot_swap') or {}
    print('%-10s chrF %.2f chrF++ %.2f BLEU %.2f  best_ep %s  unseen own-slot %s  swap %s vs %s (n=%s)' % (
        r['run'], d['chrF_char'], d['chrF++'], d['BLEU'], r.get('best_epoch'),
        ('%.3f' % pr['own_slot_unseen']) if pr else '-',
        ('%.3f' % sw['success_swap']) if sw.get('n_items') else '-',
        ('%.3f' % sw['success_self']) if sw.get('n_items') else '-', sw.get('n_items', '-')))